# Does ML beat the market? Walk-forward test on NSE stocks

Runs end to end in Google Colab (Runtime > Run all, about 2 minutes). Educational, not investment advice.

**Question:** can a gradient boosting model, fed simple price and volume features, predict tomorrow's up/down move well enough to make money after costs?

**Setup**
- 6 large NSE stocks plus the Nifty 50 index, about 10 years of daily data from Yahoo Finance (adjusted prices).
- 11 features per stock-day: returns over 1/5/10/20 days, 10 and 20-day volatility, volume change and relative volume, RSI(14) and RSI(5), distance from the 50-day average.
- Target: does the stock close higher tomorrow?
- **Walk-forward:** the model is retrained at the start of every year using only labels that were fully known by then, and trades the next year blind. First test year is 2023.
- Trading rule: hold a stock the next day if the predicted probability of up is above 0.5. Equal weight across the 6 stocks, 0.10% cost on every position change, compared with equal-weight buy-and-hold and the Nifty.
- A permutation test (shuffled labels) shows what Sharpe ratios pure luck produces.

The honest result is usually: no edge. That is the point of the notebook.

In [ ]:
!pip -q install yfinance

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import yfinance as yf
from sklearn.ensemble import HistGradientBoostingClassifier

TICKERS = ["RELIANCE.NS", "TCS.NS", "INFY.NS", "HDFCBANK.NS", "ICICIBANK.NS", "ITC.NS"]
BENCH = "^NSEI"
DATA_START = "2014-01-01"
FIRST_TEST_YEAR = 2023
COST = 0.001            # 0.10% per position change
RF_ANNUAL = 0.065
RF_DAILY = (1 + RF_ANNUAL) ** (1 / 252) - 1
plt.rcParams.update({"figure.facecolor": "#05080f", "axes.facecolor": "#070c16", "axes.edgecolor": "#1a2839",
                     "axes.labelcolor": "#bcc8d9", "text.color": "#dde6f2", "xtick.color": "#8494a9",
                     "ytick.color": "#8494a9", "grid.color": "#14202f", "axes.grid": True})

In [ ]:
def get(sym):
    df = yf.download(sym, start=DATA_START, progress=False, auto_adjust=True)
    df.columns = df.columns.get_level_values(0)
    df.index = pd.to_datetime(df.index).tz_localize(None)
    return df.dropna(subset=["Close"])

raw = {t: get(t) for t in TICKERS + [BENCH]}
print({t: (d.index[0].date(), d.index[-1].date(), len(d)) for t, d in raw.items()})

In [ ]:
def rsi(close, n):
    d = close.diff()
    up, down = d.clip(lower=0).rolling(n).mean(), (-d.clip(upper=0)).rolling(n).mean()
    return 100 - 100 / (1 + up / down)

FEATURES = [f"ret_{n}" for n in (1, 5, 10, 20)] + ["vol_10", "vol_20", "vol_chg_5", "vol_rel_20", "rsi_14", "rsi_5", "dist_ma50"]

def features(df, ticker):
    close, volume = df.Close, df.Volume.replace(0, np.nan)
    ret = close.pct_change(fill_method=None)
    x = pd.DataFrame(index=df.index)
    for n in (1, 5, 10, 20):
        x[f"ret_{n}"] = close.pct_change(n, fill_method=None)
    x["vol_10"], x["vol_20"] = ret.rolling(10).std(), ret.rolling(20).std()
    x["vol_chg_5"] = volume.pct_change(5, fill_method=None)
    x["vol_rel_20"] = volume / volume.rolling(20).mean()
    x["rsi_14"], x["rsi_5"] = rsi(close, 14), rsi(close, 5)
    x["dist_ma50"] = close / close.rolling(50).mean() - 1
    x["fwd_ret"] = ret.shift(-1)                                   # tomorrow's return = the target
    x["label_end"] = pd.Series(df.index, index=df.index).shift(-1) # date the label becomes known
    x["ticker"] = ticker
    x = x.replace([np.inf, -np.inf], np.nan).dropna(subset=FEATURES + ["fwd_ret", "label_end"])
    x["y"] = (x.fwd_ret > 0).astype(int)
    x.index.name = "date"
    return x

data = pd.concat([features(raw[t], t) for t in TICKERS])
returns = pd.DataFrame({t: raw[t].Close.pct_change(fill_method=None) for t in TICKERS}).dropna()
print(data.shape, "rows;", data.y.mean().round(3), "share of up days")

In [ ]:
def new_model():
    return HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=150,
                                          min_samples_leaf=100, l2_regularization=1.0, random_state=0)

END = data.index.max() + pd.Timedelta(days=1)

def walk_forward(data, shuffle_seed=None):
    """Retrain every Jan 1 on labels fully known before that date, predict that year."""
    edges = [pd.Timestamp(f"{y}-01-01") for y in range(FIRST_TEST_YEAR, END.year + 1)] + [END]
    out, accs = [], []
    rng = np.random.default_rng(shuffle_seed) if shuffle_seed is not None else None
    for a, b in zip(edges[:-1], edges[1:]):
        train = data[(data.index < a) & (data.label_end < a)]       # purge labels that touch the test period
        test = data[(data.index >= a) & (data.index < b)]
        if len(test) == 0: continue
        y = train.y.to_numpy()
        if rng is not None: y = rng.permutation(y)
        m = new_model().fit(train[FEATURES], y)
        t = test.copy(); t["p"] = m.predict_proba(test[FEATURES])[:, 1]
        out.append(t)
    return pd.concat(out).sort_index(kind="stable"), m

pred, last_model = walk_forward(data)
acc = ((pred.p > 0.5).astype(int) == pred.y).mean()
print(f"test rows {len(pred)}, {pred.index.min().date()} to {pred.index.max().date()}, accuracy {acc:.3f} (coin flip = 0.5, always-up = {pred.y.mean():.3f})")

In [ ]:
def run_portfolio(pred, cost=COST):
    ret = returns.loc[pred.index.min():pred.index.max()]
    pos = pred.assign(pos=(pred.p > 0.5).astype(float)).pivot(columns="ticker", values="pos")
    pos = pos.reindex(index=ret.index, columns=ret.columns).fillna(0.0)
    turnover = pos.diff().abs().fillna(pos.abs())
    strat = (pos * ret - cost * turnover).mean(axis=1)
    hold = ret.mean(axis=1).copy(); hold.iloc[0] -= cost
    return strat, hold, pos

def sharpe(r): return float((np.mean(r) - RF_DAILY) / np.std(r, ddof=1) * np.sqrt(252))

def metrics(r):
    eq = (1 + r).cumprod()
    return {"CAGR %": (eq.iloc[-1] ** (252 / len(r)) - 1) * 100, "Sharpe": sharpe(r),
            "Max DD %": ((eq / eq.cummax().clip(lower=1)) - 1).min() * 100, "Vol %": r.std() * np.sqrt(252) * 100,
            "Total %": (eq.iloc[-1] - 1) * 100}

strat, hold, pos = run_portfolio(pred)
nifty = raw[BENCH].Close.pct_change().reindex(strat.index).fillna(0)
table = pd.DataFrame({"ML strategy": metrics(strat), "Equal-weight buy&hold": metrics(hold), "Nifty 50": metrics(nifty)}).T.round(2)
table["Exposure %"] = [pos.mean().mean() * 100, 100, 100]
table.round(2)

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
for r, n, c in [(strat, "ML strategy", "#3aa4ff"), (hold, "Equal-weight buy&hold", "#e8b24a"), (nifty, "Nifty 50", "#8494a9")]:
    eq = (1 + r).cumprod(); ax[0].plot(eq, label=n, color=c, lw=1.6)
    ax[1].plot(eq / eq.cummax().clip(lower=1) - 1, color=c, lw=1)
ax[0].set_title("Out-of-sample equity (Rs 1 start)"); ax[0].legend()
ax[1].set_title("Drawdown from peak")
plt.tight_layout(); plt.show()

In [ ]:
# Is the Sharpe distinguishable from luck? Retrain 20 times on shuffled labels (destroys any real signal).
null = []
for s in range(20):
    p, _ = walk_forward(data, shuffle_seed=s)
    st, _, _ = run_portfolio(p)
    null.append(sharpe(st))
null = np.array(null)
real = sharpe(strat)
print(f"real Sharpe {real:.2f} | shuffled-label Sharpes: mean {null.mean():.2f}, range {null.min():.2f} to {null.max():.2f}")
print(f"share of shuffled runs at or above the real one: {(null >= real).mean():.0%}")
plt.hist(null, bins=10, color="#1663b8"); plt.axvline(real, color="#e8b24a", label="real model"); plt.legend()
plt.title("Sharpe of models trained on random labels"); plt.show()

In [ ]:
# What does the model look at? Permutation importance on the last year of predictions.
from sklearn.inspection import permutation_importance
last_year = pred[pred.index >= pred.index.max() - pd.Timedelta(days=365)]
imp = permutation_importance(last_model, last_year[FEATURES], last_year.y, n_repeats=5, random_state=0, scoring="accuracy")
pd.Series(imp.importances_mean, index=FEATURES).sort_values().plot.barh(color="#3aa4ff", figsize=(7, 4))
plt.title("Permutation importance (accuracy drop)"); plt.show()

## How to read this

- Accuracy near 50% means the model is guessing. Always check it against the "always up" rate.
- Compare the real Sharpe with the shuffled-label distribution. If real is inside that range, the result is noise.
- Costs matter: a model that flips positions daily pays 0.10% each time.
- Caveats: Yahoo adjusted prices (not point-in-time), today's large caps (survivorship bias), 6 stocks only, one hyperparameter setting, no tuning.

Next steps if you want to push further: more stocks, pairs trading, features beyond price and volume, longer holding periods. The web app that runs rule-based strategies lives in the same repo: **BacktestLab NSE**.